In [1]:
!pip -q install onnxruntime==1.31.1 transformers==5.22.1

In [2]:
from google.colab import drive, files
drive.mount('/content/drive')

import json
import numpy as np
import onnxruntime as ort
import pandas as pd
from sklearn.metrics import f1_score
from transformers import AutoTokenizer

DEPLOYED = '/content/drive/MyDrive/capstone-ml/models/deployed'
card = json.load(open(f'{DEPLOYED}/model_card.json'))
market = pd.read_csv('https://raw.githubusercontent.com/KeittahSewe/ratings-reputation-system/main/ml/data/market_test.csv')

tok = AutoTokenizer.from_pretrained(DEPLOYED)
so = ort.SessionOptions(); so.intra_op_num_threads = 1
sess = ort.InferenceSession(f'{DEPLOYED}/model.onnx', so, providers=['CPUExecutionProvider'])
feed = [i.name for i in sess.get_inputs()]

rows = []
for rid, text in zip(market['id'], market['text']):
    enc = tok(text, truncation=True, max_length=128, return_tensors='np')
    logits = sess.run(['logits'], {k: enc[k].astype(np.int64) for k in feed})[0][0]
    rows.append([rid, *logits])

ref = pd.DataFrame(rows, columns=['id', 'logit_negative', 'logit_neutral', 'logit_positive'])
ref['model_version'] = card['model_version']
pred = ref[['logit_negative', 'logit_neutral', 'logit_positive']].values.argmax(axis=1)
print('reviews:', len(ref))
print('correct:', int((pred == market['label'].values).sum()), 'of', len(ref))
print('macro-F1:', round(f1_score(market['label'], pred, average='macro'), 4),
      '| model card:', card['metrics']['market_macro_f1_seed_exported'])

ref.to_csv('/content/market_reference_int8.csv', index=False, float_format='%.6f')
files.download('/content/market_reference_int8.csv')

Mounted at /content/drive
reviews: 390
correct: 257 of 390
macro-F1: 0.6589 | model card: 0.6589


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>